# 01. Python + NumPy + ООП: каталог экзопланет — решения преподавателя

В этом ноутбуке мы построим маленький инструмент для анализа астрономических данных,
а затем применим его к **реальному каталогу подтверждённых экзопланет NASA Exoplanet Archive**.

**Основная идея:** сначала мы делаем часть работы «вручную» с Python и NumPy,
затем упаковываем её в собственный класс `ExoplanetDataset`. В следующем ноутбуке
похожие операции будут выполняться уже с помощью Polars.

### Что повторяем
- списки, словари, условия, циклы, функции;
- `try/except`, `enumerate`, `zip`, comprehensions;
- массивы NumPy, индексацию и boolean masks;
- `np.nan`, статистики и стандартизацию;
- основы ООП: `class`, `self`, `__init__`, `__len__`, `__repr__`, `__getitem__`,
  `@property`, `@classmethod`;
- чтение CSV;
- физический смысл некоторых параметров экзопланет.

### Уровни
- без звезды — обязательное задание;
- Дополнительное задание — дополнительное;
- Задание повышенной сложности — challenge.

> **Правило:** до раздела с NASA не используйте `pandas`, `polars` и `scikit-learn`.
> Наша цель — понять, что именно библиотеки будут делать за нас позже.

## 0. Импорты

В основной части нам нужны только стандартная библиотека Python и NumPy.

### Материалы к разделу

Перед выполнением достаточно просмотреть первые примеры; читать документацию целиком не требуется.

- [NumPy: absolute basics for beginners](https://numpy.org/doc/stable/user/absolute_beginners.html) — что такое `ndarray`, создание массивов, `shape`, индексация и основные операции.
- [Python Tutorial](https://docs.python.org/3/tutorial/) — официальный учебник Python; полезен, если нужно вспомнить базовый синтаксис.
- [NumPy documentation](https://numpy.org/doc/stable/) — главная страница документации, откуда удобно искать функцию по имени.

In [ ]:
import csv
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import urlretrieve

import numpy as np

np.set_printoptions(precision=3, suppress=True)

## 1. Разминка: маленький каталог планет

Пока без NASA. Представим, что мы уже нашли несколько планет и записали четыре признака:
название, радиус в радиусах Земли, орбитальный период в сутках и равновесную температуру.

### Материалы к разделу

- [NumPy: indexing and slicing](https://numpy.org/doc/stable/user/absolute_beginners.html#indexing-and-slicing) — обычная и булева индексация.
- [NumPy: indexing on ndarrays](https://numpy.org/doc/stable/user/basics.indexing.html) — подробная документация по индексам и boolean masks.
- [NumPy quickstart: indexing with boolean arrays](https://numpy.org/doc/stable/user/quickstart.html#indexing-with-boolean-arrays) — короткие примеры фильтрации по условиям.

Особенно обратите внимание на выражения вида `a[a < 5]` и на объединение условий через `&` и `|`.

In [ ]:
names = np.array(["Astra-1 b", "Astra-2 b", "Astra-3 b", "Astra-4 b", "Astra-5 b"])
radius = np.array([1.10, 2.40, 11.20, 0.80, 1.55])
period = np.array([365.0, 12.3, 4.1, 89.2, 42.0])
temperature = np.array([280.0, 740.0, 1300.0, 340.0, 310.0])

print(names)

### Задание 1

1. Найдите название самой большой планеты.
2. Найдите название самой горячей планеты.
3. Получите массив названий планет с радиусом меньше `2 R⊕`.
4. Получите массив названий планет с температурой от 200 до 350 K включительно.
5. Создайте словарь `period_by_name`, где ключ — название, значение — орбитальный период.

Для пунктов 1–4 постарайтесь не использовать Python-цикл.

In [ ]:
largest_name = names[np.argmax(radius)]
hottest_name = names[np.argmax(temperature)]
small_names = names[radius < 2]
temperate_names = names[(temperature >= 200) & (temperature <= 350)]
period_by_name = {name: p for name, p in zip(names, period)}

### Самопроверка

In [ ]:
assert largest_name == "Astra-3 b"
assert hottest_name == "Astra-3 b"
assert set(small_names) == {"Astra-1 b", "Astra-4 b", "Astra-5 b"}
assert set(temperate_names) == {"Astra-1 b", "Astra-4 b", "Astra-5 b"}
assert period_by_name["Astra-2 b"] == 12.3
print("OK: Задание 1 пройдено")

## 2. Генерируем собственный каталог

Создадим 1000 искусственных планет. Мы не пытаемся получить идеальную модель реальной
популяции экзопланет: этот набор нужен для тренировки NumPy и ООП.

Большая полуось `a` задаётся в а.е., масса звезды — в массах Солнца.
Период вычислим приближённо из третьего закона Кеплера:

\[
P_{\mathrm{year}} = \sqrt{\frac{a^3}{M_\star}}.
\]

Затем добавим небольшой шум измерений.

### Материалы к разделу

- [NumPy random sampling](https://numpy.org/doc/stable/reference/random/index.html) — `default_rng`, seed и генерация случайных выборок.
- [NumPy Random Generator](https://numpy.org/doc/stable/reference/random/generator.html) — методы `normal`, `uniform`, `lognormal` и другие распределения.
- [numpy.argsort](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html) — как получить индексы, которые сортируют массив.
- [NumPy beginner guide](https://numpy.org/doc/stable/user/absolute_beginners.html) — операции над массивами без Python-циклов.

Полезная идея: boolean-массив состоит из `True` и `False`; его среднее равно доле элементов, для которых условие истинно.

In [ ]:
rng = np.random.default_rng(42)
n = 1000

synthetic_names = np.array([f"SIM-{i:04d} b" for i in range(n)], dtype=object)

st_mass = np.clip(rng.normal(loc=1.0, scale=0.25, size=n), 0.15, None)
st_teff = np.clip(rng.normal(loc=5600, scale=900, size=n), 2500, 10000)

pl_orbsmax = 10 ** rng.uniform(-2.0, 0.7, size=n)
period_years_ideal = np.sqrt(pl_orbsmax**3 / st_mass)
pl_orbper = period_years_ideal * 365.25 * rng.normal(1.0, 0.02, size=n)

pl_rade = np.clip(rng.lognormal(mean=np.log(2.0), sigma=0.75, size=n), 0.2, 25)
pl_eqt = np.clip(278 * np.sqrt(np.sqrt(st_mass) / pl_orbsmax), 50, 3500)

synthetic_data = {
    "pl_name": synthetic_names,
    "pl_rade": pl_rade,
    "pl_orbper": pl_orbper,
    "pl_orbsmax": pl_orbsmax,
    "pl_eqt": pl_eqt,
    "st_mass": st_mass,
    "st_teff": st_teff,
}

{key: value.shape for key, value in synthetic_data.items()}

### Задание 2. NumPy без циклов

Для `pl_rade`:

1. найдите среднее, медиану и стандартное отклонение;
2. найдите индексы пяти планет с наименьшим радиусом;
3. посчитайте число планет с `R < 2 R⊕`;
4. посчитайте долю таких планет;
5. найдите планеты, у которых одновременно `R < 2 R⊕` и `200 <= T_eq <= 350 K`.

Дополнительное задание Для последнего пункта выведите не только количество, но и их названия.

In [ ]:
mean_radius = np.mean(pl_rade)
median_radius = np.median(pl_rade)
std_radius = np.std(pl_rade)

five_smallest_idx = np.argsort(pl_rade)[:5]
n_small = np.sum(pl_rade < 2)
share_small = np.mean(pl_rade < 2)

candidate_mask = (pl_rade < 2) & (pl_eqt >= 200) & (pl_eqt <= 350)
candidate_names = synthetic_names[candidate_mask]

## 3. «Грязные» данные

Настоящие научные каталоги содержат пропуски и неоднородные измерения.
Добавим несколько проблем в искусственные данные и научимся их находить.

### Материалы к разделу

- [NumPy statistics](https://numpy.org/doc/stable/reference/routines.statistics.html) — `mean`, `median`, `std`, `nanmean`, `nanmedian`, `nanstd`.
- [numpy.nanstd](https://numpy.org/doc/stable/reference/generated/numpy.nanstd.html) — пример статистики с игнорированием `NaN`.
- [Stack Overflow: ignore NaN in NumPy functions](https://stackoverflow.com/questions/66289714/ignore-nan-in-numpy-functions) — почему обычные `mean` и `std` возвращают `NaN` и какие функции использовать вместо них.
- [Python: Errors and Exceptions](https://docs.python.org/3/tutorial/errors.html) — пригодится позже при обработке неправильных типов и значений.

Не путайте `np.nan` с нулём: `NaN` означает отсутствие или неопределённость числового значения.

In [ ]:
dirty_data = {key: value.copy() for key, value in synthetic_data.items()}

dirty_data["pl_rade"][[17, 213, 581]] = np.nan
dirty_data["pl_eqt"][[42, 512]] = np.nan

dirty_data["pl_rade"][341] = 500.0
dirty_data["pl_eqt"][56] = -100.0

### Задание 3. Вспомогательные функции

Реализуйте:

- `count_missing(x)` — количество `NaN`;
- `zscore(x)` — стандартизация с игнорированием `NaN`;
- `outlier_mask(x, threshold=3)` — маска объектов, для которых `|z| > threshold`.

\[
z = \frac{x-\mu}{\sigma}.
\]

In [ ]:
def count_missing(x):
    x = np.asarray(x, dtype=float)
    return int(np.isnan(x).sum())

def zscore(x):
    x = np.asarray(x, dtype=float)
    mean = np.nanmean(x)
    std = np.nanstd(x)
    if std == 0:
        raise ValueError("Нельзя стандартизовать константный массив")
    return (x - mean) / std

def outlier_mask(x, threshold=3):
    return np.abs(zscore(x)) > threshold

In [ ]:
assert count_missing(dirty_data["pl_rade"]) == 3

z = zscore(np.array([1.0, 2.0, 3.0, np.nan]))
assert np.isclose(np.nanmean(z), 0.0)
assert np.isclose(np.nanstd(z), 1.0)

assert outlier_mask(dirty_data["pl_rade"]).dtype == bool
assert outlier_mask(dirty_data["pl_rade"]).shape == (n,)

print("OK: Вспомогательные функции работают")

## 4. Пишем собственный `ExoplanetDataset`

Внутри `data` хранится словарь `{"column": np.array(...)}`. Все столбцы должны иметь одинаковую длину.

Реализуйте `__init__`, `__len__`, `__repr__`, `columns`, `shape`, `__getitem__`,
`get_row`, `filter`, `add_column`, статистические методы, `unique`, `value_counts`,
`standardize`, а затем Дополнительное задание `from_csv`.

**Важно:** `filter` и `add_column` не должны менять исходный объект.

### Материалы к разделу

- [Python Tutorial: Classes](https://docs.python.org/3/tutorial/classes.html) — официально про классы, экземпляры, методы и атрибуты.
- [Real Python: Object-Oriented Programming in Python](https://realpython.com/python3-object-oriented-programming/) — более подробное объяснение ООП на примерах.
- [Real Python: Classes in Python, video](https://realpython.com/videos/classes-python/) — короткое видео о классах и экземплярах.
- [Python built-in functions: `classmethod`](https://docs.python.org/3/library/functions.html#classmethod) — что именно получает `cls`.
- [Stack Overflow: class methods as alternative constructors](https://stackoverflow.com/questions/73569542/how-class-methods-can-be-alternative-constructors-as-they-just-return-us-objects) — полезно перед реализацией `from_csv`.

При работе над классом не пытайтесь реализовать всё сразу: сначала добейтесь прохождения тестов для `__init__`, `__len__`, `columns` и `shape`, затем переходите к остальным методам.

In [ ]:
class ExoplanetDataset:
    def __init__(self, data):
        if not isinstance(data, dict) or len(data) == 0:
            raise ValueError("data должен быть непустым словарём")
        arrays = {name: np.asarray(values).copy() for name, values in data.items()}
        lengths = {len(values) for values in arrays.values()}
        if len(lengths) != 1:
            raise ValueError("Все столбцы должны иметь одинаковую длину")
        self.data = arrays

    def __len__(self):
        return len(next(iter(self.data.values())))

    def __repr__(self):
        return f"ExoplanetDataset(n_rows={len(self)}, n_columns={len(self.columns)})"

    @property
    def columns(self):
        return list(self.data.keys())

    @property
    def shape(self):
        return (len(self), len(self.columns))

    def __getitem__(self, column):
        if column not in self.data:
            raise KeyError(f"Неизвестный столбец: {column}")
        return self.data[column]

    def get_row(self, index):
        if index < -len(self) or index >= len(self):
            raise IndexError("Индекс строки вне диапазона")
        return {name: values[index] for name, values in self.data.items()}

    def filter(self, mask):
        mask = np.asarray(mask, dtype=bool)
        if mask.shape != (len(self),):
            raise ValueError("Маска должна иметь длину, равную числу строк")
        return ExoplanetDataset({name: values[mask] for name, values in self.data.items()})

    def add_column(self, name, values):
        values = np.asarray(values)
        if values.shape != (len(self),):
            raise ValueError("Новый столбец должен иметь длину len(dataset)")
        new_data = {key: value.copy() for key, value in self.data.items()}
        new_data[name] = values.copy()
        return ExoplanetDataset(new_data)

    def _numeric(self, column):
        try:
            return self[column].astype(float)
        except (ValueError, TypeError):
            raise TypeError(f"Столбец {column!r} не является числовым")

    def mean(self, column):
        return float(np.nanmean(self._numeric(column)))

    def median(self, column):
        return float(np.nanmedian(self._numeric(column)))

    def std(self, column):
        return float(np.nanstd(self._numeric(column)))

    def count_missing(self, column):
        values = self[column]
        if np.issubdtype(values.dtype, np.number):
            return int(np.isnan(values.astype(float)).sum())
        return int(sum(
            value is None or (isinstance(value, str) and value.strip() == "")
            for value in values
        ))

    def describe(self, column):
        x = self._numeric(column)
        valid = x[~np.isnan(x)]
        if len(valid) == 0:
            return {
                "count": 0, "missing": len(x), "mean": np.nan, "std": np.nan,
                "min": np.nan, "median": np.nan, "max": np.nan,
            }
        return {
            "count": int(len(valid)),
            "missing": int(np.isnan(x).sum()),
            "mean": float(np.mean(valid)),
            "std": float(np.std(valid)),
            "min": float(np.min(valid)),
            "median": float(np.median(valid)),
            "max": float(np.max(valid)),
        }

    def unique(self, column):
        values = self[column]
        clean = [
            value for value in values
            if value is not None and not (isinstance(value, str) and value.strip() == "")
        ]
        return np.unique(np.asarray(clean, dtype=object))

    def value_counts(self, column):
        counts = {}
        for value in self[column]:
            if value is None or (isinstance(value, str) and value.strip() == ""):
                continue
            counts[value] = counts.get(value, 0) + 1
        return counts

    def standardize(self, column):
        x = self._numeric(column)
        mean = np.nanmean(x)
        std = np.nanstd(x)
        if std == 0:
            raise ValueError("Нельзя стандартизовать константный столбец")
        return (x - mean) / std

    @staticmethod
    def _parse_column(values):
        parsed = []
        numeric = True

        for value in values:
            value = value.strip()
            if value == "":
                parsed.append(np.nan)
                continue
            try:
                parsed.append(float(value))
            except ValueError:
                numeric = False
                break

        if numeric:
            return np.asarray(parsed, dtype=float)

        return np.asarray(
            [value.strip() if value.strip() != "" else None for value in values],
            dtype=object,
        )

    @classmethod
    def from_csv(cls, filename):
        with open(filename, newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            rows = list(reader)
            fieldnames = reader.fieldnames

        if not rows:
            raise ValueError("CSV-файл пуст")

        raw_columns = {
            name: [row[name] for row in rows]
            for name in fieldnames
        }
        data = {
            name: cls._parse_column(values)
            for name, values in raw_columns.items()
        }
        return cls(data)

### Тесты класса на искусственных данных

In [ ]:
ds = ExoplanetDataset(dirty_data)

assert len(ds) == 1000
assert ds.shape == (1000, 7)
assert "pl_rade" in ds.columns
assert ds["pl_rade"].shape == (1000,)
assert ds.get_row(0)["pl_name"] == "SIM-0000 b"

print(ds)
print("OK: Базовая структура класса работает")

In [ ]:
small_ds = ds.filter(ds["pl_rade"] < 2)

assert isinstance(small_ds, ExoplanetDataset)
assert len(small_ds) < len(ds)
assert len(ds) == 1000

ds2 = ds.add_column("dummy_feature", np.arange(len(ds)))
assert "dummy_feature" in ds2.columns
assert "dummy_feature" not in ds.columns

print("OK: filter и add_column работают")

In [ ]:
assert ds.count_missing("pl_rade") == 3

desc = ds.describe("pl_rade")
expected_keys = {"count", "missing", "mean", "std", "min", "median", "max"}
assert set(desc) == expected_keys
assert desc["missing"] == 3

standardized = ds.standardize("st_mass")
assert np.isclose(np.nanmean(standardized), 0.0)
assert np.isclose(np.nanstd(standardized), 1.0)

print("OK: Статистические методы работают")

### Дополнительное задание Задание 4. ООП — объясните словами

1. Что хранится в состоянии объекта `ExoplanetDataset`?
2. Почему `dataset["pl_rade"]` начинает работать после реализации `__getitem__`?
3. Почему `filter` возвращает новый объект?
4. Чем `@classmethod from_csv` удобен как альтернативный конструктор?

**Возможный ответ:** состояние — `self.data`; квадратные скобки вызывают `__getitem__`;
новый объект уменьшает побочные эффекты; `from_csv` создаёт объект класса непосредственно
из другого представления данных.

## 5. Настоящие данные: NASA Exoplanet Archive

Используем `Planetary Systems Composite Parameters (pscomppars)`.
Для учебной работы берём 15 колонок.

Документация:
- https://exoplanetarchive.ipac.caltech.edu/docs/TAP/usingTAP.html
- https://exoplanetarchive.ipac.caltech.edu/docs/API_PS_columns.html

**Преподавателю:** включите скачивание один раз и затем раздайте
`nasa_exoplanets.csv` вместе с ноутбуком. Тогда у всех учеников будут одинаковые данные.

### Материалы к разделу

- [Python `csv` module](https://docs.python.org/3/library/csv.html) — официальная документация по `csv.reader` и `csv.DictReader`.
- [Real Python: Reading CSVs with Python's csv module, video](https://realpython.com/videos/reading-csvs-pythons-csv-module/) — короткий разбор `DictReader`.
- [NASA Exoplanet Archive: TAP documentation](https://exoplanetarchive.ipac.caltech.edu/docs/TAP/usingTAP.html) — как программно запрашиваются таблицы архива.
- [NASA Exoplanet Archive: PS/PSCompPars columns](https://exoplanetarchive.ipac.caltech.edu/docs/API_PS_columns.html) — физический смысл полей каталога.

При чтении CSV помните: `csv.DictReader` возвращает значения как строки. Поэтому числовые колонки необходимо преобразовать отдельно.

In [ ]:
NASA_COLUMNS = [
    "pl_name", "hostname", "discoverymethod", "disc_year",
    "pl_orbper", "pl_orbsmax", "pl_rade", "pl_bmasse",
    "pl_eqt", "pl_insol", "st_teff", "st_rad", "st_mass",
    "sy_dist", "sy_pnum",
]

NASA_CSV_PATH = Path("nasa_exoplanets.csv")

query = "select " + ",".join(NASA_COLUMNS) + " from pscomppars"
params = urlencode({"query": query, "format": "csv"})
NASA_URL = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync?" + params

DOWNLOAD_NASA_DATA = False

if DOWNLOAD_NASA_DATA:
    urlretrieve(NASA_URL, NASA_CSV_PATH)
    print("Сохранено:", NASA_CSV_PATH.resolve())
else:
    print("Автоскачивание выключено.")
    print("Для создания снимка преподавателем: DOWNLOAD_NASA_DATA = True")

### Задание 5. Загрузите NASA-каталог

Реализуйте `from_csv`, положите `nasa_exoplanets.csv` рядом с ноутбуком и загрузите его.

In [ ]:
if NASA_CSV_PATH.exists():
    nasa = ExoplanetDataset.from_csv(NASA_CSV_PATH)
    print(nasa)
    print(nasa.columns)
    print(nasa.get_row(0))
else:
    print("Файл nasa_exoplanets.csv не найден.")

## 6. EDA без pandas и Polars

### Задание 6

1. Сколько планет в снимке?
2. Для каких пяти числовых колонок доля пропусков максимальна?
3. Найдите медианные радиус и период.
4. Найдите пять планет с наименьшим известным радиусом.
5. Найдите пять планет с самым коротким известным периодом.

### Материалы к разделу

- [NumPy boolean indexing](https://numpy.org/doc/stable/user/basics.indexing.html#boolean-array-indexing) — фильтрация строк по маске.
- [numpy.argsort](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html) — поиск объектов с минимальными или максимальными значениями.
- [NumPy statistics](https://numpy.org/doc/stable/reference/routines.statistics.html) — функции для описательной статистики.
- [NASA: Exoplanet Detection Methods](https://www.nasa.gov/universe/exoplanets/exoplanet-detection-methods/) — астрономический контекст методов обнаружения.

Для поиска пяти минимальных значений обычно удобно сначала убрать `NaN`, затем применить `np.argsort(...)[:5]`.

In [ ]:
if NASA_CSV_PATH.exists():
    numeric_columns = [
        "disc_year", "pl_orbper", "pl_orbsmax", "pl_rade", "pl_bmasse",
        "pl_eqt", "pl_insol", "st_teff", "st_rad", "st_mass", "sy_dist", "sy_pnum"
    ]

    missing_share = {
        col: nasa.count_missing(col) / len(nasa)
        for col in numeric_columns
    }
    top_missing = sorted(missing_share.items(), key=lambda item: item[1], reverse=True)[:5]

    print("Планет:", len(nasa))
    print("Топ-5 по пропускам:", top_missing)
    print("Медианный радиус:", nasa.median("pl_rade"))
    print("Медианный период:", nasa.median("pl_orbper"))

    valid_r = ~np.isnan(nasa["pl_rade"])
    idx = np.argsort(nasa["pl_rade"][valid_r])[:5]
    print(list(zip(nasa["pl_name"][valid_r][idx], nasa["pl_rade"][valid_r][idx])))

    valid_p = ~np.isnan(nasa["pl_orbper"])
    idx = np.argsort(nasa["pl_orbper"][valid_p])[:5]
    print(list(zip(nasa["pl_name"][valid_p][idx], nasa["pl_orbper"][valid_p][idx])))

### Задание 7. Методы открытия

Получите уникальные методы открытия, частоты и отсортируйте их по убыванию.
Дополнительное задание Определите самый частый метод среди планет с `disc_year >= 2015`.

In [ ]:
if NASA_CSV_PATH.exists():
    counts = nasa.value_counts("discoverymethod")
    for method, count in sorted(counts.items(), key=lambda x: x[1], reverse=True):
        print(f"{method:30s} {count}")

    recent = nasa.filter(nasa["disc_year"] >= 2015)
    recent_counts = recent.value_counts("discoverymethod")
    print("После 2015:", max(recent_counts.items(), key=lambda x: x[1]))

### Задание 8. Учебный поиск «похожих на Землю по двум параметрам»

Отберите:
- `0.8 <= pl_rade <= 1.5`;
- `200 <= pl_eqt <= 350`.

Выведите название, радиус и температуру.

> Это не критерий обитаемости, а только упражнение на фильтрацию.

In [ ]:
if NASA_CSV_PATH.exists():
    r = nasa["pl_rade"]
    t = nasa["pl_eqt"]

    mask = (
        ~np.isnan(r) & ~np.isnan(t)
        & (r >= 0.8) & (r <= 1.5)
        & (t >= 200) & (t <= 350)
    )

    candidates = nasa.filter(mask)
    for i in range(len(candidates)):
        row = candidates.get_row(i)
        print(row["pl_name"], row["pl_rade"], row["pl_eqt"])

## 7. Физика + NumPy

### Задание 9. Относительная средняя плотность Дополнительное задание

\[
\frac{\rho_p}{\rho_\oplus}
=
\frac{M_p/M_\oplus}{(R_p/R_\oplus)^3}.
\]

Вычислите `relative_density`, добавьте её в новый датасет и найдите пять максимальных
известных значений. Объясните, почему экстремумы надо интерпретировать осторожно.

### Материалы к разделу

- [NASA JPL: Exploring Exoplanets with Kepler](https://www.jpl.nasa.gov/edu/resources/lesson-plan/exploring-exoplanets-with-kepler/) — школьное объяснение третьего закона Кеплера и его применения к экзопланетам.
- [NASA Science: Orbits and Kepler's Laws](https://science.nasa.gov/solar-system/orbits-and-keplers-laws/) — физический смысл трёх законов Кеплера.
- [NASA Exoplanet Archive: column definitions](https://exoplanetarchive.ipac.caltech.edu/docs/API_PS_columns.html) — единицы `pl_orbper`, `pl_orbsmax`, `pl_bmasse`, `pl_rade`, `st_mass`.
- [NumPy indexing](https://numpy.org/doc/stable/user/basics.indexing.html) — создание маски только для строк, где нужные величины известны.

Перед подстановкой в формулу обязательно проверьте единицы: период в каталоге задан в сутках, а в используемой форме закона Кеплера нужен период в годах.

In [ ]:
if NASA_CSV_PATH.exists():
    mass = nasa["pl_bmasse"].astype(float)
    r = nasa["pl_rade"].astype(float)

    valid = ~np.isnan(mass) & ~np.isnan(r) & (mass > 0) & (r > 0)
    relative_density = np.full(len(nasa), np.nan)
    relative_density[valid] = mass[valid] / r[valid]**3

    nasa_with_density = nasa.add_column("relative_density", relative_density)

    valid_density = ~np.isnan(relative_density)
    idx = np.argsort(relative_density[valid_density])[-5:][::-1]
    print(list(zip(
        nasa["pl_name"][valid_density][idx],
        relative_density[valid_density][idx]
    )))

### Задание 10. Проверяем третий закон Кеплера Задание повышенной сложности

\[
P_{\mathrm{year}}^2 \approx \frac{a^3}{M_\star}.
\]

Для строк с известными `pl_orbper`, `pl_orbsmax`, `st_mass`:
переведите период в годы, вычислите предсказанный период и относительную ошибку

\[
error = \frac{|P_{\mathrm{pred}} - P_{\mathrm{obs}}|}{P_{\mathrm{obs}}}.
\]

Найдите медианную ошибку и пять объектов с максимальной ошибкой.
Затем предложите минимум две причины расхождений.

In [ ]:
if NASA_CSV_PATH.exists():
    p_days = nasa["pl_orbper"].astype(float)
    a = nasa["pl_orbsmax"].astype(float)
    m_star = nasa["st_mass"].astype(float)

    valid = (
        ~np.isnan(p_days) & ~np.isnan(a) & ~np.isnan(m_star)
        & (p_days > 0) & (a > 0) & (m_star > 0)
    )

    p_obs = p_days[valid] / 365.25
    p_pred = np.sqrt(a[valid]**3 / m_star[valid])
    error = np.abs(p_pred - p_obs) / p_obs

    print("Медианная относительная ошибка:", np.median(error))

    worst = np.argsort(error)[-5:][::-1]
    names_valid = nasa["pl_name"][valid]
    for i in worst:
        print(names_valid[i], error[i])

Возможные причины: измерительные неопределённости; значения в композитной таблице могут
происходить из разных публикаций и не быть строго самосогласованными; приближение
пренебрегает массой планеты; сложные многотельные системы могут давать дополнительные эффекты.

## 8. Финальное мини-исследование

В 5–8 предложениях ответьте:

1. Какие признаки заполнены хуже всего?
2. Какой метод открытия доминирует?
3. Чем отличается полнота данных по радиусам и массам?
4. Какие объекты прошли учебный фильтр по радиусу и температуре?
5. Насколько хорошо выполняется проверка третьего закона Кеплера?
6. Назовите два ограничения анализа.

### Материалы к разделу

- [NASA Exoplanet Archive](https://exoplanetarchive.ipac.caltech.edu/) — исходный научный каталог, с которым вы работаете.
- [NASA Exoplanet Resource Guide](https://www.nasa.gov/stem-content/exoplanet-resource-guide/) — материалы о методах поиска и свойствах экзопланет.
- [NumPy statistics](https://numpy.org/doc/stable/reference/routines.statistics.html) — если нужно перепроверить используемые статистики.

Финальный текст должен опираться на полученные числа. Разделяйте наблюдение в данных и физическую интерпретацию: например, большая доля транзитных открытий не означает, что во Вселенной большинство планет обязательно обнаружимо именно этим методом.

**Комментарий преподавателю:** численные результаты зависят от даты снимка NASA, поэтому
эталон содержит код, а не фиксированные ответы. Для проверки используйте один общий CSV.

## 9. Что нужно сдать

Заполненный `.ipynb`, где проходят тесты, выполнен анализ NASA и написан итоговый вывод.

В следующем ноутбуке мы сравним собственный класс с **Polars** и добавим
визуализацию через **Matplotlib**.

### Материалы к разделу

- [Jupyter Notebook: running code](https://jupyter-notebook.readthedocs.io/en/stable/examples/Notebook/Running%20Code.html) — запуск ячеек и перезапуск kernel.
- [Jupyter Notebook basic workflow](https://github.com/jupyter/notebook/blob/main/docs/source/notebook.md#basic-workflow) — как устроена работа с состоянием ноутбука.

Перед сдачей рекомендуется перезапустить kernel и выполнить все ячейки сверху вниз. Это позволяет обнаружить ситуации, когда код случайно зависит от переменной, созданной в другой последовательности запуска.